# Introduction
__[markdown cheatsheet](https://www.ibm.com/docs/en/watson-studio-local/1.2.3?topic=notebooks-markdown-jupyter-cheatsheet)__,
__[yt video1](https://youtu.be/PK3wL7DXuuw?si=MS0sf4e_MKWl-Eyd)__
__[automata project](https://merveyavuz.github.io/Automata/Project/index.html)__
https://en.wikipedia.org/wiki/Synchronizing_word
https://youtu.be/YtD2KGRdn3s?si=6_937iQ2KPB8tOJr
https://www.sciencedirect.com/topics/computer-science/deterministic-automaton

### To do list
- Spellcheck
- Part 1
- Finish introduction

This report will explore Deterministic Finite Automata (DFA), an important concept in theoretical computer science and mathematics. A DFA is a finite state machine that takes in string of symbols and accepts or rejects the string based on the sequence of states. DFAs have many practical applications including; recognising formal languages, malware detection, text processing, user interface and compilers. The theoretical foundations were layed by the work of Alan Turing in the 1930s, creating the Turing machine which helped the development of finite state machines which became essential for the first computer engineers.
<br><br>

We will be specifically exploring how to identify a *synchronizing* DFA which is a special type of DFA as they contain a *reset word* in their alphabet. The Cerny Conjecture states that (n-1)^2 is the largest length of a reset word for a DFA with n states. The conjecture made by Jan Cerny in 1969 still remains an unsolved problem in Computer science. We will discover that synchronizing DFAs correspond to a transformation semigroup, a set of functions that is closed under composition and contains a constant map (if synchronizing). We will also discover that forming a pair digraph for a DFA will help in identifying if a given DFA is synchronizing. 
<br><br>

We will also be exploring *isomorphisms* between two DFAs that have various levels of strictness.

# Part 1: Synchronization

In [1]:
#example DFA
Q = {"q0","q1","q2"}
sigma = {"0","1"}
tau = {
    ("q0","0"):"q1",
    ("q0","1"):"q0",
    ("q1","0"):"q1",
    ("q1","1"):"q2",
    ("q2","0"):"q1",
    ("q2","1"):"q0",
}
qs = "q0"
F = {"q2"}
dfa1 = (Q,sigma,tau,qs,F)

def dfa_accept_word(A,word):
    Q, sigma, tau, qs, F = A
    state = qs

    for letter in word:
        state = tau[(state,letter)]

    if state in F:
        return "accepted"
    else:
        return "rejected"

dfa_accept_word(dfa1,"101")

'accepted'

1a) To test if a given DFA is synchronizing we will use A BFS algorithm.
- We want to go through each letter in sigma,
- add it to a queue to check if the corresponding function is constant for all states.
- Then, try composition of these letters and add these to the queue

In [1]:
def breadth_first_synch_test(A):
    #Start by converting the 5-tuple into each of its elements
    Q, sigma, tau, qs, F = A
 
    #defining the corresponding transformation ta to the letter a
    def transformation_of_letter(a):
        return{q: tau[(q,a)] for q in Q}

    #defining composition of transformations
    def composition_f_o_g(f,g):
        return {q: f[g[q]] for q in Q}
    
    transformations_checked = set()
    queue = [transformation_of_letter(a) for a in sigma]
    
    while queue:
        #remove first transformation from queue
        f = queue.pop(0)
        #convert that transformation to a tuple of the form
        #((q0,f(q0)),(q1,f(q1)),...,(qn,f(qn)))
       
        ftup = tuple(f.items())

        #if the transformation has been checked, go back to start of while loop
        if ftup in transformations_checked:
            continue

        #add transformation to the checked set
        transformations_checked.add(ftup)
        
        #look at the set of image values for the transformation, if |im(f)| = 1, then its constant
        if len(set(image for state,image in ftup)) == 1:
            return True
        #composing the transformations for each letter a with the f being currently examined: ta(f(q))
        for a in sigma:
            ta = transformation_of_letter(a)
            composedtransformation = composition_f_o_g(ta, f)
            queue.append(composedtransformation)
            
    return False

breadth_first_synch_test(dfa1)

NameError: name 'dfa1' is not defined

1b)

In [3]:
def breadth_second_synch_test(A):
    #Start by converting the 5-tuple into each of its elements
    Q, sigma, tau, qs, F = A
    '''
    Using lots of code from breadth_first_synch_test
    Not including any of the local functions as will not be used
    '''

    #function to find image of subset under specified letter
    def image(f, a):
        return{tau[(q,a)] for q in f}

    #this time we will store subsets of states instead of transformations
    subsets_of_states_checked = set()

    
    list_queue = list(Q)
    list_queue.sort()
    queue = [tuple(list_queue)]


    while queue:
        #remove first subset from queue
        f = queue.pop(0)

        #if the current subset has been checked, go back to start of while loop
        if f in subsets_of_states_checked:
            continue

        subsets_of_states_checked.add(f)

        #if we find subset with only one state, DFA is synchronising 
        if len(f) == 1:
            return True

        #compute image of subset under each letter, sort it and convert to tuple
        for a in sigma:
            im = image(f, a)
            im = list(im)
            im.sort()
            im = tuple(im)

            #if a new subset, add to the queue
            if im not in subsets_of_states_checked:
                queue.append(im)
    #if no subset with 1 element found, DFA not synchronising
    return False
    
    
breadth_second_synch_test(dfa1)

True

1c)

In [4]:
def pair_graph_synch_test(A):
    #Start by converting the 5-tuple into each of its elements
    Q, sigma, tau, qs, F = A

In [ ]:
import random

def generate_dfa(n, m, f_size):
    """
    Generate a random DFA A = (Q, sigma, tau, qs, F) with:
        Q = {0, 1, ..., n-1}
        sigma = {0, 1, ..., m-1}
        tau: dict with keys (state, symbol) -> next state
        qs = 0 (start state)
        F = set of f_size accepting states

    Ensures every state is reachable from the start state.
    """
    if f_size < 0 or f_size > n:
        raise ValueError("f_size must be between 0 and n")

    # Track available symbols per state (to avoid collisions when building the spanning tree)
    available = [set(range(m)) for _ in range(n)]

    # Transition function: dict (state, symbol) -> next_state
    tau = {}

    # Build a spanning tree from state 0 to all other states
    reachable = [0]          # states already in the tree
    remaining = set(range(1, n))   # states not yet added

    while remaining:
        # Pick a random state from reachable that still has free symbols
        candidates = [s for s in reachable if available[s]]
        u = random.choice(candidates)

        # Pick a random unused symbol for u
        a = random.choice(list(available[u]))

        # Pick a random target from the remaining states
        v = random.choice(list(remaining))

        # Assign the transition
        tau[(u, a)] = v
        available[u].remove(a)

        # Update sets
        reachable.append(v)
        remaining.remove(v)

    # Fill all remaining transitions arbitrarily
    for u in range(n):
        for a in range(m):
            if (u, a) not in tau:
                tau[(u, a)] = random.randint(0, n-1)

    # Choose accepting states
    F = set(random.sample(range(n), f_size))

    # Q and sigma as sets (as described)
    Q = set(range(n))
    sigma = set(range(m))
    qs = 0

    return Q, sigma, tau, qs, F